In [2]:
import os
import numpy as np
import librosa
import soundfile as sf
import noisereduce as nr  # Библиотека для шумоподавления
from pathlib import Path
from tqdm import tqdm

# ============================================================================
# КОНФИГУРАЦИЯ
# ============================================================================

# Новый путь к исходной папке со всеми файлами (без классов)
INPUT_PATH = "D:/Аудио выборка/Аудиосегменты/Данные_усталости"

# Путь для сохранения обработанных данных
PROCESSED_PATH = "D:/Аудио выборка/Аудиосегменты/Предобработанные_данные"

# Параметры обработки
SR = 16000  # целевая частота дискретизации
TARGET_DURATION = 1.0  # целевая длительность в секундах
TARGET_LEN = int(SR * TARGET_DURATION)  # 16000 отсчётов
NOISE_REDUCTION_ENABLED = True  # Включаем шумоподавление
NORMALIZATION_TYPE = 'soft'  # 'soft' или 'hard'
NORMALIZATION_RMS_LEVEL = 0.1  # целевой RMS уровень для мягкой нормализации

# ============================================================================
# ФУНКЦИИ ПРЕДОБРАБОТКИ
# ============================================================================

def reduce_noise_stationary(audio, sr, noise_sample_duration=0.2):
    """
    Шумоподавление стационарного шума с помощью библиотеки noisereduce.
    """
    noise_sample_len = int(noise_sample_duration * sr)
    if len(audio) > noise_sample_len:
        noise_sample = audio[:noise_sample_len]
        audio_clean = nr.reduce_noise(
            y=audio,
            sr=sr,
            y_noise=noise_sample,
            stationary=True,
            prop_decrease=0.8,
            n_std_thresh_stationary=1.5
        )
        return audio_clean
    return audio


def normalize_audio_soft(audio, target_rms=0.1):
    """
    Мягкая нормализация: приведение к целевому RMS уровню
    Сохраняет относительные различия в громкости
    """
    rms = np.sqrt(np.mean(audio ** 2))
    if rms > 0:
        gain = target_rms / rms
        gain = min(gain, 3.0)
        return audio * gain
    return audio


def adjust_duration(audio, target_len, sr):
    """
    Приведение аудио к целевой длительности
    Обрезка или дополнение нулями
    """
    if len(audio) > target_len:
        start = (len(audio) - target_len) // 2
        return audio[start:start + target_len]
    elif len(audio) < target_len:
        return np.pad(audio, (0, target_len - len(audio)), mode='constant')
    return audio


def preprocess_audio(file_path, target_len=TARGET_LEN, sr=SR):
    """
    Полный пайплайн предобработки аудиофайла.
    """
    try:
        audio, orig_sr = librosa.load(file_path, sr=sr)
        
        if np.max(np.abs(audio)) < 0.01:
            return None
        
        if NOISE_REDUCTION_ENABLED:
            audio = reduce_noise_stationary(audio, sr)
        
        if NORMALIZATION_TYPE == 'hard':
            if np.max(np.abs(audio)) > 0:
                audio = audio / np.max(np.abs(audio))
        else:  # soft
            audio = normalize_audio_soft(audio, target_rms=NORMALIZATION_RMS_LEVEL)
        
        audio = adjust_duration(audio, target_len, sr)
        return audio
        
    except Exception as e:
        print(f"Ошибка при обработке {file_path}: {e}")
        return None


def process_folder_single(input_path, output_path, target_len=TARGET_LEN, sr=SR):
    """
    Обрабатывает все аудиофайлы в одной папке и сохраняет обработанные копии в output_path.
    Классы не учитываются.
    """
    os.makedirs(output_path, exist_ok=True)
    
    input_files = list(Path(input_path).glob("*.wav"))
    print(f"\nОбработка файлов: найдено {len(input_files)} файлов")
    
    processed_count = 0
    error_count = 0
    
    for file_path in tqdm(input_files, desc="Обработка файлов"):
        audio_processed = preprocess_audio(str(file_path), target_len, sr)
        
        if audio_processed is not None:
            output_file = os.path.join(output_path, file_path.name)
            sf.write(output_file, audio_processed, sr)
            processed_count += 1
        else:
            error_count += 1
    
    print(f"  Успешно обработано: {processed_count}, ошибок: {error_count}")
    return processed_count, error_count


def preprocess_all_data_single(input_path, output_path):
    """
    Обрабатывает все аудиофайлы в одной папке (без деления на классы).
    """
    print("="*60)
    print("ПРЕДОБРАБОТКА АУДИОДАННЫХ")
    print("="*60)
    print(f"Целевая частота дискретизации: {SR} Гц")
    print(f"Целевая длительность: {TARGET_DURATION} сек ({TARGET_LEN} отсчётов)")
    print(f"Шумоподавление: {'Включено' if NOISE_REDUCTION_ENABLED else 'Выключено'}")
    print(f"Тип нормализации: {NORMALIZATION_TYPE.upper()}")
    
    if os.path.exists(input_path):
        processed, errors = process_folder_single(input_path, output_path)
    else:
        print(f"\nПапка не найдена - {input_path}")
        processed, errors = 0, 0
    
    print("\n" + "="*60)
    print("ОБРАБОТКА ЗАВЕРШЕНА")
    print("="*60)
    print(f"Всего обработано файлов: {processed}")
    print(f"Ошибок: {errors}")
    print(f"Результаты сохранены в: {output_path}")
    
    return processed, errors


# ============================================================================
# ЗАПУСК ПРЕДОБРАБОТКИ
# ============================================================================

if __name__ == "__main__":
    total_processed, total_errors = preprocess_all_data_single(INPUT_PATH, PROCESSED_PATH)

ПРЕДОБРАБОТКА АУДИОДАННЫХ
Целевая частота дискретизации: 16000 Гц
Целевая длительность: 1.0 сек (16000 отсчётов)
Шумоподавление: Включено
Тип нормализации: SOFT

Обработка файлов: найдено 3211 файлов


Обработка файлов: 100%|██████████| 3211/3211 [01:39<00:00, 32.16it/s]

  Успешно обработано: 3211, ошибок: 0

ОБРАБОТКА ЗАВЕРШЕНА
Всего обработано файлов: 3211
Ошибок: 0
Результаты сохранены в: D:/Аудио выборка/Аудиосегменты/Предобработанные_данные


### Дополнительные уставшие

In [1]:
import os
import numpy as np
import librosa
import soundfile as sf
import noisereduce as nr  # Библиотека для шумоподавления
from pathlib import Path
from tqdm import tqdm

# ============================================================================
# КОНФИГУРАЦИЯ
# ============================================================================

# Параметры обработки
SR = 16000  # целевая частота дискретизации
TARGET_DURATION = 1.0  # целевая длительность в секундах
TARGET_LEN = int(SR * TARGET_DURATION)  # 16000 отсчётов
NOISE_REDUCTION_ENABLED = True  # Включаем шумоподавление
NORMALIZATION_TYPE = 'soft'  # 'soft' или 'hard'
NORMALIZATION_RMS_LEVEL = 0.1  # целевой RMS уровень для мягкой нормализации

# ============================================================================
# ФУНКЦИИ ПРЕДОБРАБОТКИ
# ============================================================================

def reduce_noise_stationary(audio, sr, noise_sample_duration=0.2):
    """
    Шумоподавление стационарного шума с помощью библиотеки noisereduce.
    """
    noise_sample_len = int(noise_sample_duration * sr)
    if len(audio) > noise_sample_len:
        noise_sample = audio[:noise_sample_len]
        audio_clean = nr.reduce_noise(
            y=audio,
            sr=sr,
            y_noise=noise_sample,
            stationary=True,
            prop_decrease=0.8,
            n_std_thresh_stationary=1.5
        )
        return audio_clean
    return audio


def normalize_audio_soft(audio, target_rms=0.1):
    """
    Мягкая нормализация: приведение к целевому RMS уровню
    Сохраняет относительные различия в громкости
    """
    rms = np.sqrt(np.mean(audio ** 2))
    if rms > 0:
        gain = target_rms / rms
        gain = min(gain, 3.0)
        return audio * gain
    return audio


def adjust_duration(audio, target_len, sr):
    """
    Приведение аудио к целевой длительности
    Обрезка или дополнение нулями
    """
    if len(audio) > target_len:
        start = (len(audio) - target_len) // 2
        return audio[start:start + target_len]
    elif len(audio) < target_len:
        return np.pad(audio, (0, target_len - len(audio)), mode='constant')
    return audio


def preprocess_audio(file_path, target_len=TARGET_LEN, sr=SR):
    """
    Полный пайплайн предобработки аудиофайла.
    """
    try:
        audio, orig_sr = librosa.load(file_path, sr=sr)
        
        if np.max(np.abs(audio)) < 0.01:
            return None
        
        if NOISE_REDUCTION_ENABLED:
            audio = reduce_noise_stationary(audio, sr)
        
        if NORMALIZATION_TYPE == 'hard':
            if np.max(np.abs(audio)) > 0:
                audio = audio / np.max(np.abs(audio))
        else:  # soft
            audio = normalize_audio_soft(audio, target_rms=NORMALIZATION_RMS_LEVEL)
        
        audio = adjust_duration(audio, target_len, sr)
        return audio
        
    except Exception as e:
        print(f"Ошибка при обработке {file_path}: {e}")
        return None


def process_folder_single(input_path, output_path, target_len=TARGET_LEN, sr=SR):
    """
    Обрабатывает все аудиофайлы в одной папке и сохраняет обработанные копии в output_path.
    Классы не учитываются.
    """
    os.makedirs(output_path, exist_ok=True)
    
    input_files = list(Path(input_path).glob("*.wav"))
    print(f"\nОбработка файлов: найдено {len(input_files)} файлов")
    
    processed_count = 0
    error_count = 0
    
    for file_path in tqdm(input_files, desc="Обработка файлов"):
        audio_processed = preprocess_audio(str(file_path), target_len, sr)
        
        if audio_processed is not None:
            output_file = os.path.join(output_path, file_path.name)
            sf.write(output_file, audio_processed, sr)
            processed_count += 1
        else:
            error_count += 1
    
    print(f"  Успешно обработано: {processed_count}, ошибок: {error_count}")
    return processed_count, error_count


def preprocess_all_data_single(input_path, output_path):
    """
    Обрабатывает все аудиофайлы в одной папке (без деления на классы).
    """
    print("="*60)
    print("ПРЕДОБРАБОТКА АУДИОДАННЫХ")
    print("="*60)
    print(f"Целевая частота дискретизации: {SR} Гц")
    print(f"Целевая длительность: {TARGET_DURATION} сек ({TARGET_LEN} отсчётов)")
    print(f"Шумоподавление: {'Включено' if NOISE_REDUCTION_ENABLED else 'Выключено'}")
    print(f"Тип нормализации: {NORMALIZATION_TYPE.upper()}")
    
    if os.path.exists(input_path):
        processed, errors = process_folder_single(input_path, output_path)
    else:
        print(f"\nПапка не найдена - {input_path}")
        processed, errors = 0, 0
    
    print("\n" + "="*60)
    print("ОБРАБОТКА ЗАВЕРШЕНА")
    print("="*60)
    print(f"Всего обработано файлов: {processed}")
    print(f"Ошибок: {errors}")
    print(f"Результаты сохранены в: {output_path}")
    
    return processed, errors


# Новый путь к исходной папке со всеми файлами (без классов)
INPUT_PATH = "C:/Users/user/Downloads/positive_fragments_sh2"

# Путь для сохранения обработанных данных
PROCESSED_PATH = "D:/Аудио выборка/Аудиосегменты/Предобработанные_данные"

# ============================================================================
# ЗАПУСК ПРЕДОБРАБОТКИ
# ============================================================================

if __name__ == "__main__":
    total_processed, total_errors = preprocess_all_data_single(INPUT_PATH, PROCESSED_PATH)

ПРЕДОБРАБОТКА АУДИОДАННЫХ
Целевая частота дискретизации: 16000 Гц
Целевая длительность: 1.0 сек (16000 отсчётов)
Шумоподавление: Включено
Тип нормализации: SOFT

Обработка файлов: найдено 35 файлов


Обработка файлов: 100%|██████████| 35/35 [00:04<00:00,  7.86it/s]

  Успешно обработано: 35, ошибок: 0

ОБРАБОТКА ЗАВЕРШЕНА
Всего обработано файлов: 35
Ошибок: 0
Результаты сохранены в: D:/Аудио выборка/Аудиосегменты/Предобработанные_данные


In [3]:
import os
folder_path=r"C:\Users\user\Downloads\positive_fragments_sh"
for filename in os.listdir(folder_path):
    if filename.endswith("_01.wav"):
        new_filename = filename.replace("_01.wav", "_1.wav")
        old_path = os.path.join(folder_path, filename)
        new_path = os.path.join(folder_path, new_filename)
        os.rename(old_path, new_path)
        print(f"Переименован: {filename} -> {new_filename}")

Переименован: shelkoplyas_0605_0014_9.80-10.80_01.wav -> shelkoplyas_0605_0014_9.80-10.80_1.wav
Переименован: shelkoplyas_0605_0023_16.10-17.10_01.wav -> shelkoplyas_0605_0023_16.10-17.10_1.wav
Переименован: shelkoplyas_0605_0050_35.00-36.00_01.wav -> shelkoplyas_0605_0050_35.00-36.00_1.wav
Переименован: shelkoplyas_0605_0069_48.30-49.30_01.wav -> shelkoplyas_0605_0069_48.30-49.30_1.wav
Переименован: shelkoplyas_0605_0082_57.40-58.40_01.wav -> shelkoplyas_0605_0082_57.40-58.40_1.wav
Переименован: shelkoplyas_0605_0097_67.90-68.90_01.wav -> shelkoplyas_0605_0097_67.90-68.90_1.wav
Переименован: shelkoplyas_0605_1_0007_4.90-5.90_01.wav -> shelkoplyas_0605_1_0007_4.90-5.90_1.wav
Переименован: shelkoplyas_0605_1_0014_9.80-10.80_01.wav -> shelkoplyas_0605_1_0014_9.80-10.80_1.wav
Переименован: shelkoplyas_0605_1_0023_16.10-17.10_01.wav -> shelkoplyas_0605_1_0023_16.10-17.10_1.wav
Переименован: shelkoplyas_0605_1_0050_35.00-36.00_01.wav -> shelkoplyas_0605_1_0050_35.00-36.00_1.wav
Переименован